# Проектирование экспериментов и интерпретация моделей на базе Hugging Face Transformers

## Содержание

1. Введение
2. Часть 1. Объектно-ориентированный интерфейс эксперимента
   - 1.1. Постановка задачи
   - 1.2. Словарь терминов ООП
   - 1.3. Инструмент `dataclass`
   - 1.4. Инструмент `ABC` и абстрактные методы
   - 1.5. Наследование и полиморфизм в интерфейсе эксперимента
   - 1.6. Реализация конкретного эксперимента
   - 1.7. Реестр экспериментов
   - 1.8. Задания и вопросы
3. Часть 2. Интерпретация предсказаний: LIME
   - 2.1. Назначение метода
   - 2.2. Словарь терминов
   - 2.3. Подготовка функции предсказания
   - 2.4. Получение объяснения
   - 2.5. Ограничения метода
   - 2.6. Задания и вопросы
4. Часть 3. Интерпретация внутренних представлений: Captum
   - 3.1. Назначение метода
   - 3.2. Словарь терминов
   - 3.3. Вычисление атрибуции
   - 3.4. Агрегация и интерпретация
   - 3.5. Специфика применения к Transformer-моделям
   - 3.6. Задания и вопросы
5. Часть 4. Итоговый отчёт
6. Заключение
7. Домашнее задание
8. Приложение. Сводная таблица терминов

## 1. Введение

В ходе выполнения предыдущих заданий студенты строят модели классификации текста и достигают приемлемых показателей точности. При этом возникают два системных затруднения.

**Затруднение первое — организация экспериментального кода.** Типичное решение оформляется как функция с длинным списком аргументов. Добавление новой архитектуры или нового гиперпараметра требует правки всех мест вызова. Сравнение нескольких конфигураций в едином цикле становится затруднительным, поскольку функция вынуждена содержать ветвления для выбора архитектуры.

**Затруднение второе — интерпретация результатов.** Метрика accuracy представляет собой агрегированное число. Она описывает поведение модели на уровне корпуса и не даёт ответа на вопрос о причинах конкретного ошибочного предсказания. Тимлид, получающий отчёт с одной метрикой, лишён возможности оценить, на какие признаки опирается модель и как изменится её поведение при смене архитектуры.

Пособие построено на библиотеке Hugging Face Transformers, которая предоставляет готовые компоненты для тонкой настройки предобученных моделей. Использование библиотеки позволяет сосредоточиться на методологии экспериментирования, а не на реализации циклов обучения.

**Структура пособия.**

| Часть | Тема | Инструменты |
|---|---|---|
| 1 | Объектно-ориентированный интерфейс эксперимента | `dataclass`, `ABC`, наследование, полиморфизм |
| 2 | Интерпретация на уровне входных признаков | LIME |
| 3 | Интерпретация на уровне внутренних представлений | Captum, Integrated Gradients |
| 4 | Итоговый отчёт | Таблица, визуализация, текст |
| 5 | Домашнее задание | Самостоятельная реализация и анализ |

**Условные обозначения в примерах кода.**

| Обозначение | Значение |
|---|---|
| **«До»** | Код, который типично пишут без использования изучаемого инструмента |
| **«После»** | Код, переписанный с использованием инструмента |
| **«Что делает»** | Построчный разбор функции или метода |
| **«Изолированный запуск»** | Минимальный воспроизводимый пример, не требующий полного пайплайна |
| **«Допишите»** | Фрагмент с пропусками, который студент заполняет самостоятельно |

In [ ]:
# !pip install -q transformers datasets lime captum accelerate

import os
import random
import time
from dataclasses import dataclass, field
from abc import ABC, abstractmethod
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
)
from datasets import load_dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device = {device}")
print(f"torch = {torch.__version__}")

In [ ]:
dataset = load_dataset("ag_news")
train_raw = dataset["train"].shuffle(seed=42).select(range(4000))
eval_raw = dataset["test"].shuffle(seed=42).select(range(1000))

class_names = ["World", "Sports", "Business", "Sci/Tech"]

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_fn(example):
    return tokenizer(example["text"], truncation=True, max_length=128)

train_ds = train_raw.map(tokenize_fn, batched=True)
eval_ds = eval_raw.map(tokenize_fn, batched=True)

train_ds = train_ds.rename_column("label", "labels")
eval_ds = eval_ds.rename_column("label", "labels")
train_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
eval_ds.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

collator = DataCollatorWithPadding(tokenizer=tokenizer)

print(train_ds)
print(eval_ds)

## Часть 1. Объектно-ориентированный интерфейс эксперимента

### 1.1. Постановка задачи

Экспериментальная работа в машинном обучении состоит в систематическом сравнении конфигураций. **Конфигурация** — это набор параметров, полностью определяющий один запуск: архитектура модели, гиперпараметры обучения, параметры предобработки данных. Для проведения сравнения необходимо, чтобы все конфигурации обрабатывались единообразно: один и тот же код отвечает за загрузку данных, инициализацию модели, обучение и оценку, а различия между запусками задаются только значениями параметров.

Функциональная организация нарушает это требование по двум причинам.

| Причина | Следствие |
|---|---|
| Сигнатура функции содержит все параметры | Добавление параметра требует изменения всех мест вызова |
| Функция содержит ветвления для выбора архитектуры | Разные архитектуры требуют разных классов моделей, ветвление растёт |

Объектно-ориентированный подход устраняет оба недостатка за счёт разделения ответственности: конфигурация инкапсулируется в объекте данных, а различия в поведении реализуются через наследование.

**Пример «до»: типичная функциональная организация.**

In [ ]:
def run_experiment(name, model_name, lr, epochs, batch_size, max_len,
                   use_fp16, warmup_steps, weight_decay, seed):
    # 1. загрузка данных
    # 2. токенизация
    # 3. выбор модели через if/elif
    if model_name.startswith("distilbert"):
        model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=4)
    elif model_name.startswith("roberta"):
        model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=4)
    else:
        raise ValueError("unknown model")
    # 4. обучение
    # 5. оценка
    return {"name": name, "model_name": model_name}

Каждое новое требование — early stopping, F1-метрика, сохранение чекпоинтов — добавляет аргумент в сигнатуру и ветвление в тело функции.

**Пример «после»: интерфейс на классах.**

```python
configs = [
    ExperimentConfig(name="distilbert", model_name="distilbert-base-uncased"),
    ExperimentConfig(name="roberta", model_name="roberta-base", learning_rate=1e-5),
]

results = []
for cfg in configs:
    experiment_class = EXPERIMENT_REGISTRY[cfg.name]
    results.append(experiment_class(cfg).run(train_ds, eval_ds))
```

Этот фрагмент заработает после того, как ниже будут определены `ExperimentConfig`, `EXPERIMENT_REGISTRY`, `DistilBertExperiment` и `RobertaExperiment`.

Добавление архитектуры сводится к созданию подкласса и записи в реестр. Цикл запуска не изменяется.

### 1.2. Словарь терминов ООП

Перед изучением инструментов необходимо зафиксировать значения терминов, которые используются далее.

| Термин | Определение |
|---|---|
| **Класс** | Шаблон, описывающий набор атрибутов (данных) и методов (функций), которыми обладают объекты этого класса. |
| **Объект (экземпляр)** | Конкретная сущность, созданная по шаблону класса. Объект хранит собственные значения атрибутов. |
| **Атрибут** | Именованное поле данных, связанное с объектом или классом. В `ExperimentConfig` атрибутами являются `name`, `model_name`, `learning_rate`. |
| **Метод** | Функция, определённая внутри класса и принимающая первым аргументом ссылку на объект (`self`). |
| **Конструктор** | Метод `__init__`, вызываемый при создании объекта. Присваивает начальные значения атрибутам. |
| **Наследование** | Механизм, при котором класс-потомок получает атрибуты и методы класса-родителя и может добавлять собственные или переопределять унаследованные. |
| **Базовый класс (родитель)** | Класс, от которого наследуют другие классы. В пособии — `BaseExperiment`. |
| **Подкласс (потомок)** | Класс, наследующий базовому. В пособии — `DistilBertExperiment`, `RobertaExperiment`. |
| **Переопределение метода** | Реализация метода в подклассе с тем же именем, что и в базовом классе. Вызов метода на объекте подкласса выполняет реализацию подкласса. |
| **Полиморфизм** | Свойство, при котором код, работающий с объектами базового класса, корректно работает и с объектами любого подкласса без знания о конкретном типе. |
| **Абстрактный метод** | Метод, объявленный в базовом классе без реализации. Подкласс обязан предоставить реализацию. |
| **Абстрактный базовый класс (ABC)** | Класс, который не может быть инстанцирован, пока все его абстрактные методы не реализованы в подклассе. |
| **Инкапсуляция** | Объединение данных и методов, работающих с этими данными, в одном классе, с ограничением прямого доступа к внутреннему состоянию. |
| **Контракт** | Набор обязательств, которые класс обязуется выполнять. В пособии контракт задаётся абстрактными методами базового класса. |

**Изолированный запуск: проверим разницу между классом и объектом.**

In [ ]:
class Dog:
    def __init__(self, name):
        self.name = name

    def bark(self):
        return f"{self.name} says woof"

# Dog — класс (шаблон)
# Dog("Rex") — объект (экземпляр)
d = Dog("Rex")
print(type(Dog))     # <class 'type'> — это класс
print(type(d))       # <class '__main__.Dog'> — это объект
print(d.bark())      # Rex says woof

**Построчный разбор.**

| Строка | Что делает |
|---|---|
| `class Dog:` | Объявляет класс `Dog` |
| `def __init__(self, name):` | Конструктор, вызываемый при создании объекта |
| `self.name = name` | Создаёт атрибут `name` у объекта |
| `def bark(self):` | Метод, доступный всем объектам класса `Dog` |
| `d = Dog("Rex")` | Создаёт объект, вызывает `__init__` |
| `d.bark()` | Вызывает метод на объекте; `self` ссылается на `d` |

### 1.3. Инструмент `dataclass`

**Назначение.** Декоратор `@dataclass` из стандартной библиотеки Python автоматически генерирует методы `__init__`, `__repr__` и `__eq__` на основе аннотаций типов полей класса. Для задачи конфигурации эксперимента это означает, что разработчик объявляет поля с типами и значениями по умолчанию, а Python обеспечивает корректную инициализацию и читаемое представление объекта.

**Как работает.** Декоратор анализирует аннотации класса и создаёт метод `__init__`, принимающий параметры в порядке объявления полей. Поля со значениями по умолчанию становятся необязательными параметрами. Метод `__repr__` возвращает строку с именем класса и значениями всех полей, что упрощает отладку.

**Роль `__post_init__`.** Согласно документации Python, если класс определён с декоратором `@dataclass` и содержит метод `__post_init__`, сгенерированный `__init__` вызывает этот метод после присвоения значений всем полям. Это позволяет выполнить валидацию значений в момент создания объекта, до начала вычислений.

**Пример «до»: конфигурация через словарь.**

In [ ]:
config = {
    "name": "distilbert",
    "model_name": "distilbert-base-uncased",
    "learning_rate": 2e-5,
    "num_epochs": 3,
    "batch_size": 16,
}

# Опечатка в имени поля не обнаруживается до обращения
try:
    print(config["learnign_rate"])
except KeyError as e:
    print(f"Поймано: KeyError: {e}")

**Пример «после»: конфигурация через `dataclass`.**

In [ ]:
@dataclass
class ExperimentConfig:
    name: str
    model_name: str = "distilbert-base-uncased"
    learning_rate: float = 2e-5
    num_epochs: int = 3
    batch_size: int = 16
    max_length: int = 128

    def __post_init__(self):
        if self.learning_rate <= 0:
            raise ValueError(
                f"learning_rate must be positive, got {self.learning_rate}"
            )
        if self.num_epochs < 1:
            raise ValueError(
                f"num_epochs must be at least 1, got {self.num_epochs}"
            )

**Что делает `__post_init__`.** Вызывается автоматически после `__init__`. Проверяет два условия: скорость обучения положительна, число эпох не меньше единицы. При нарушении возбуждает `ValueError` с указанием конкретного значения, которое вызвало ошибку.

**Изолированный запуск: проверка валидации.**

In [ ]:
# Корректный конфиг
cfg = ExperimentConfig(name="baseline")
print(cfg)

# Некорректный lr
try:
    ExperimentConfig(name="bad", learning_rate=-1)
except ValueError as e:
    print(f"Поймано: {e}")

# Некорректное число эпох
try:
    ExperimentConfig(name="bad", num_epochs=0)
except ValueError as e:
    print(f"Поймано: {e}")

**Сравнение подходов.**

| Аспект | Словарь `dict` | `dataclass` |
|---|---|---|
| Подсказки в IDE | Отсутствуют | Присутствуют |
| Валидация значений | Вручную в месте использования | В `__post_init__` |
| Опечатка в имени поля | `KeyError` при обращении | `AttributeError` при обращении |
| Читаемость | `cfg["lr"]` | `cfg.learning_rate` |
| Значения по умолчанию | Требуют `dict.get` | Объявляются в полях |
| Сравнение объектов | По ссылке | По значениям полей (генерируется `__eq__`) |

**Изолированный запуск: сравнение двух конфигов.**

In [ ]:
a = ExperimentConfig(name="baseline")
b = ExperimentConfig(name="baseline")
print(a == b)   # True — dataclass сравнивает по значениям полей
print(a is b)   # False — это разные объекты в памяти

**Допишите код самостоятельно.** Добавьте в `ExperimentConfig` поле `warmup_ratio: float = 0.1` и проверку в `__post_init__`, что значение лежит в диапазоне `[0.0, 1.0]`.

In [ ]:
@dataclass
class ExperimentConfigExtended:
    name: str
    model_name: str = "distilbert-base-uncased"
    learning_rate: float = 2e-5
    num_epochs: int = 3
    batch_size: int = 16
    max_length: int = 128
    warmup_ratio: float = 0.1

    def __post_init__(self):
        if self.learning_rate <= 0:
            raise ValueError(
                f"learning_rate must be positive, got {self.learning_rate}"
            )
        if self.num_epochs < 1:
            raise ValueError(
                f"num_epochs must be at least 1, got {self.num_epochs}"
            )
        if not (0.0 <= self.warmup_ratio <= 1.0):
            raise ValueError(
                f"warmup_ratio must be in [0, 1], got {self.warmup_ratio}"
            )

# Проверка
print(ExperimentConfigExtended(name="ok"))
try:
    ExperimentConfigExtended(name="bad", warmup_ratio=1.5)
except ValueError as e:
    print(f"Поймано: {e}")

### 1.4. Инструмент `ABC` и абстрактные методы

**Назначение.** Модуль `abc` предоставляет механизм абстрактных базовых классов. Абстрактный базовый класс задаёт контракт: он объявляет методы, которые обязан предоставить каждый подкласс, и запрещает создание экземпляров до тех пор, пока все такие методы не реализованы.

**Как работает.** Класс наследует `ABC`. Методы, помеченные декоратором `@abstractmethod`, не имеют реализации (или имеют, но подкласс обязан её переопределить). При попытке создания экземпляра подкласса интерпретатор Python проверяет, реализованы ли все абстрактные методы. Если хотя бы один не реализован, возбуждается исключение `TypeError` с указанием нереализованных методов.

**Пример «до»: базовый класс без контракта.**

In [ ]:
class BaseExperimentLoose:
    def build_model(self):
        raise NotImplementedError("subclass must implement build_model")

    def run(self, train_ds, eval_ds):
        raise NotImplementedError("subclass must implement run")


class BrokenExperimentLoose(BaseExperimentLoose):
    pass


# Создание проходит, ошибка возникает только при вызове
exp = BrokenExperimentLoose()         # OK
try:
    exp.run(None, None)              # NotImplementedError
except NotImplementedError as e:
    print(f"Поймано: {e}")

Проблема: объект создаётся успешно, ошибка обнаруживается лишь при вызове метода. Если `run` вызывается через час после создания, отладка усложняется.

**Пример «после»: базовый класс с контрактом `ABC`.**

In [ ]:
class BaseExperiment(ABC):
    def __init__(self, config: ExperimentConfig):
        self.config = config
        self.result = None

    @abstractmethod
    def build_model(self):
        """Создать модель для классификации."""
        ...

    @abstractmethod
    def run(self, train_dataset, eval_dataset):
        """Выполнить обучение и оценку, вернуть словарь с результатами."""
        ...

**Что делает `@abstractmethod`.** Регистрирует метод в множестве абстрактных методов класса. При создании экземпляра метакласс `ABCMeta` проверяет, что каждый абстрактный метод переопределён в конкретном классе. Если нет — возбуждается `TypeError`.

**Изолированный запуск: проверка контракта.**

In [ ]:
class BrokenExperiment(BaseExperiment):
    pass

try:
    exp = BrokenExperiment(ExperimentConfig(name="broken"))
except TypeError as e:
    print(f"Поймано: {e}")

**Изолированный запуск: минимальный корректный подкласс.**

In [ ]:
class MinimalExperiment(BaseExperiment):
    def build_model(self):
        return None

    def run(self, train_dataset, eval_dataset):
        self.result = {"name": self.config.name, "eval_accuracy": 0.0}
        return self.result


exp = MinimalExperiment(ExperimentConfig(name="minimal"))
print(exp.run(None, None))

**Сравнение подходов.**

| Аспект | `NotImplementedError` | `ABC` + `@abstractmethod` |
|---|---|---|
| Момент обнаружения ошибки | При вызове метода | При создании объекта |
| Проверка полноты реализации | Отсутствует | Автоматическая |
| Читаемость контракта | Неявная | Явная через декоратор |
| Возможность частичной реализации | Да | Только для неабстрактных методов |

**Допишите код самостоятельно.** Добавьте в `BaseExperiment` третий абстрактный метод `evaluate(self, eval_dataset)`, возвращающий словарь метрик. Реализуйте его в `MinimalExperiment`.

In [ ]:
class BaseExperimentWithEval(BaseExperiment):
    @abstractmethod
    def evaluate(self, eval_dataset):
        """Вернуть словарь метрик на отложенной выборке."""
        ...


class MinimalExperimentWithEval(BaseExperimentWithEval):
    def build_model(self):
        return None

    def run(self, train_dataset, eval_dataset):
        self.result = self.evaluate(eval_dataset)
        return self.result

    def evaluate(self, eval_dataset):
        return {"name": self.config.name, "eval_accuracy": 0.0}


exp = MinimalExperimentWithEval(ExperimentConfig(name="minimal_eval"))
print(exp.run(None, None))

### 1.5. Наследование и полиморфизм в интерфейсе эксперимента

**Наследование** позволяет подклассу получить атрибуты и методы базового класса. В пособии подклассы `DistilBertExperiment` и `RobertaExperiment` наследуют `BaseExperiment`, получая конструктор, сохраняющий конфигурацию, и обязательство реализовать `build_model` и `run`.

**Полиморфизм** обеспечивает единообразную обработку объектов разных подклассов. Цикл запуска вызывает `experiment.run(train_dataset, eval_dataset)` и не проверяет, к какому классу принадлежит объект. Если объект создан по шаблону `DistilBertExperiment`, выполняется реализация `run` из этого подкласса; если по шаблону `RobertaExperiment` — из другого.

**Схема отношений.**

```
BaseExperiment (ABC)
├── build_model()  [abstract]
├── run()          [abstract]
│
├── DistilBertExperiment
│   ├── build_model()  → AutoModelForSequenceClassification("distilbert-base-uncased")
│   └── run()          → Trainer(...)
│
└── RobertaExperiment
    ├── build_model()  → AutoModelForSequenceClassification("roberta-base")
    └── run()          → Trainer(...)
```

**Изолированный запуск: демонстрация полиморфизма на простом примере.**

In [ ]:
class Animal(ABC):
    @abstractmethod
    def sound(self):
        ...


class Cat(Animal):
    def sound(self):
        return "meow"


class Dog(Animal):
    def sound(self):
        return "woof"


# Полиморфный вызов: цикл не знает, кто именно внутри
for animal in [Cat(), Dog(), Cat()]:
    print(animal.sound())

**Построчный разбор.**

| Строка | Что делает |
|---|---|
| `class Animal(ABC):` | Объявляет абстрактный базовый класс |
| `@abstractmethod def sound(self): ...` | Задаёт контракт: подкласс обязан реализовать |
| `class Cat(Animal):` | Наследует `Animal`, обязуется реализовать `sound` |
| `for animal in [Cat(), Dog(), Cat()]:` | Создаёт список объектов разных классов |
| `animal.sound()` | Вызывает метод; Python выбирает реализацию по фактическому классу объекта |

**Что делает `animal.sound()`.** При вызове метода Python ищет его в классе объекта, а не в классе переменной. Это и есть полиморфизм: одна и та же строка кода выполняет разные реализации в зависимости от фактического типа объекта.

**Допишите код самостоятельно.** Добавьте класс `Cow(Animal)` с методом `sound`, возвращающим `"moo"`. Убедитесь, что цикл из примера выше работает с ним без изменений.

In [ ]:
class Cow(Animal):
    def sound(self):
        return "moo"


for animal in [Cat(), Dog(), Cow()]:
    print(animal.sound())

### 1.6. Реализация конкретного эксперимента

Подкласс `DistilBertExperiment` реализует оба абстрактных метода.

**Метод `build_model`.** Загружает предобученную модель через `AutoModelForSequenceClassification` с указанием числа классов. Класс `AutoModelForSequenceClassification` из библиотеки Transformers автоматически определяет архитектуру по имени модели и добавляет классификационную голову поверх предобученного энкодера.

**Метод `run`.** Создаёт объект `Trainer` из библиотеки Transformers, передавая ему модель, аргументы обучения, наборы данных и функцию вычисления метрик, затем выполняет обучение и оценку.

In [ ]:
class DistilBertExperiment(BaseExperiment):
    def build_model(self):
        return AutoModelForSequenceClassification.from_pretrained(
            self.config.model_name,
            num_labels=4
        )

    def run(self, train_dataset, eval_dataset):
        model = self.build_model()
        training_args = TrainingArguments(
            output_dir=f"./results/{self.config.name}",
            learning_rate=self.config.learning_rate,
            num_train_epochs=self.config.num_epochs,
            per_device_train_batch_size=self.config.batch_size,
            per_device_eval_batch_size=self.config.batch_size,
            eval_strategy="epoch",
            save_strategy="no",
            logging_steps=50,
            report_to="none",
            seed=42,
        )
        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            data_collator=collator,
        )
        trainer.train()
        metrics = trainer.evaluate()
        self.result = {"name": self.config.name, **metrics}
        return self.result

**Что делает `run`.** Последовательно выполняет пять шагов.

| Шаг | Действие | Результат |
|---|---|---|
| 1 | Вызов `build_model()` | Загруженная модель с классификационной головой |
| 2 | Создание `TrainingArguments` | Объект с параметрами обучения |
| 3 | Создание `Trainer` | Объект, связывающий модель, данные и параметры |
| 4 | `trainer.train()` | Обученная модель |
| 5 | `trainer.evaluate()` | Словарь с метриками (`eval_loss`, `eval_accuracy`) |

**Что делает `self.result = {"name": ..., **metrics}`.** Объединяет имя эксперимента с метриками в один словарь. Оператор `**` распаковывает словарь `metrics` в новый словарь. Результат сохраняется в атрибуте объекта для последующего доступа.

**Изолированный запуск: `build_model` без обучения.**

In [ ]:
cfg = ExperimentConfig(name="smoke_test", model_name="distilbert-base-uncased")
exp = DistilBertExperiment(cfg)
model = exp.build_model()

print(type(model))
print(model.config.num_labels)
print(sum(p.numel() for p in model.parameters()))

**Изолированный запуск: `TrainingArguments` без `Trainer`.**

In [ ]:
cfg = ExperimentConfig(name="args_test", learning_rate=3e-5, num_epochs=2)
exp = DistilBertExperiment(cfg)
print(exp.config.learning_rate)
print(exp.config.num_epochs)

**Соответствие полей конфигурации и аргументов обучения.**

| Поле `ExperimentConfig` | Аргумент `TrainingArguments` | Назначение |
|---|---|---|
| `learning_rate` | `learning_rate` | Скорость обучения оптимизатора |
| `num_epochs` | `num_train_epochs` | Число проходов по обучающему набору |
| `batch_size` | `per_device_train_batch_size`, `per_device_eval_batch_size` | Размер мини-батча |
| `name` | `output_dir` | Каталог для сохранения результатов |

**Роль `TrainingArguments`.** Класс `TrainingArguments` в библиотеке Transformers выполняет ту же роль, что и `ExperimentConfig` в данном пособии: он инкапсулирует параметры обучения в объекте с валидацией и значениями по умолчанию. Использование собственного класса конфигурации в дополнение к `TrainingArguments` оправдано тем, что `ExperimentConfig` содержит параметры, специфичные для серии экспериментов (имя запуска, выбор модели), и транслирует их в соответствующие поля `TrainingArguments` при создании тренера.

**Допишите код самостоятельно.** Реализуйте подкласс `RobertaExperiment`, отличающийся только значением `model_name` по умолчанию. Проверьте, что `build_model` возвращает модель с корректным числом классов.

In [ ]:
class RobertaExperiment(BaseExperiment):
    def build_model(self):
        return AutoModelForSequenceClassification.from_pretrained(
            self.config.model_name,
            num_labels=4
        )

    def run(self, train_dataset, eval_dataset):
        model = self.build_model()
        training_args = TrainingArguments(
            output_dir=f"./results/{self.config.name}",
            learning_rate=self.config.learning_rate,
            num_train_epochs=self.config.num_epochs,
            per_device_train_batch_size=self.config.batch_size,
            per_device_eval_batch_size=self.config.batch_size,
            eval_strategy="epoch",
            save_strategy="no",
            logging_steps=50,
            report_to="none",
            seed=42,
        )
        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            data_collator=collator,
        )
        trainer.train()
        metrics = trainer.evaluate()
        self.result = {"name": self.config.name, **metrics}
        return self.result


cfg = ExperimentConfig(name="roberta_smoke", model_name="roberta-base")
exp = RobertaExperiment(cfg)
m = exp.build_model()
print(type(m))
print(m.config.num_labels)

### 1.7. Реестр экспериментов

**Назначение.** Реестр устраняет необходимость в условных операторах внутри цикла запуска. Реестр представляет собой словарь, отображающий строковый ключ (тип модели) на класс эксперимента.

**Пример «до»: ветвление в цикле.**

```python
results = []
for cfg in configs:
    if cfg.name == "distilbert":
        exp = DistilBertExperiment(cfg)
    elif cfg.name == "roberta":
        exp = RobertaExperiment(cfg)
    elif cfg.name == "bert":
        exp = BertExperiment(cfg)
    else:
        raise ValueError(f"unknown experiment: {cfg.name}")
    results.append(exp.run(train_ds, eval_ds))
```

Каждая новая архитектура добавляет ветвление. Цикл знает о конкретных классах, что нарушает принцип полиморфизма.

**Пример «после»: реестр.**

In [ ]:
EXPERIMENT_REGISTRY = {
    "distilbert": DistilBertExperiment,
    "roberta": RobertaExperiment,
}

configs = [
    ExperimentConfig(name="distilbert", model_name="distilbert-base-uncased"),
    ExperimentConfig(name="roberta", model_name="roberta-base", learning_rate=1e-5),
]

# Иллюстрация цикла без фактического запуска обучения:
for cfg in configs:
    experiment_class = EXPERIMENT_REGISTRY[cfg.name]
    experiment = experiment_class(cfg)
    print(f"would run: {cfg.name} via {experiment_class.__name__}")

**Что делает `EXPERIMENT_REGISTRY[cfg.name]`.** Возвращает класс (не объект) по строковому ключу. Затем `experiment_class(cfg)` создаёт объект этого класса, вызывая конструктор `BaseExperiment.__init__`, унаследованный подклассом.

**Изолированный запуск: реестр без обучения.**

In [ ]:
EXPERIMENT_REGISTRY = {"distilbert": DistilBertExperiment}

# Получаем класс по ключу
cls = EXPERIMENT_REGISTRY["distilbert"]
print(cls)

# Создаём объект
cfg = ExperimentConfig(name="distilbert", model_name="distilbert-base-uncased")
exp = cls(cfg)
print(type(exp))
print(exp.config.name)

# Неизвестный ключ
try:
    EXPERIMENT_REGISTRY["unknown"]
except KeyError as e:
    print(f"Поймано: {e}")

**Свойства решения.**

| Свойство | Реализация |
|---|---|
| Цикл не содержит ветвлений по типу модели | Выбор класса выполняется через словарь |
| Добавление архитектуры | Создание подкласса и запись в реестр |
| Единообразие обработки | Все объекты наследуют `BaseExperiment` |
| Обработка неизвестного ключа | `KeyError` с указанием отсутствующего ключа |

**Допишите код самостоятельно.** Добавьте в реестр ключ `"bert"` и реализуйте `BertExperiment`, отличающийся от `DistilBertExperiment` только значением `model_name` по умолчанию (`"bert-base-uncased"`). Проверьте, что цикл из раздела 1.7 работает с тремя архитектурами без изменений.

In [ ]:
class BertExperiment(DistilBertExperiment):
    def __init__(self, config):
        super().__init__(config)
        if config.model_name == "distilbert-base-uncased":
            self.config.model_name = "bert-base-uncased"


EXPERIMENT_REGISTRY.update({"bert": BertExperiment})
print(list(EXPERIMENT_REGISTRY.keys()))

### 1.8. Задания и вопросы

**Задание 1.1.** Изучите документацию класса `Trainer` и определите, какой параметр конструктора позволяет передать функцию для вычисления метрик помимо стандартных потерь. Реализуйте подкласс `BaseExperiment`, который использует этот параметр для вычисления макро-усреднённой F1-меры в дополнение к accuracy.

*Ориентир для поиска.* В документации Transformers найдите страницу `Trainer` и раздел, посвящённый параметру `compute_metrics`. Функция должна принимать объект `EvalPrediction` и возвращать словарь с названиями метрик и их значениями.

**Задание 1.2.** В цикле запуска из раздела 1.7 используется `EXPERIMENT_REGISTRY[cfg.name]`. Предложите альтернативный способ выбора, при котором цикл не зависит от того, что ключ реестра совпадает с `cfg.name`. Реализуйте предложение.

**Задание 1.3.** Реализуйте декоратор `@register_experiment("name")`, который автоматически добавляет класс в `EXPERIMENT_REGISTRY` при объявлении. Подсказка: декоратор принимает строку и возвращает функцию, принимающую класс и возвращающую его же после записи в реестр.

**Вопрос 1.1 (выбор ответа).** Какой механизм обеспечивает невозможность создания экземпляра класса `BaseExperiment`?

- A. Декоратор `@dataclass`
- B. Декоратор `@abstractmethod` в сочетании с наследованием `ABC`
- C. Метод `__post_init__`
- D. Метод `__init__`

**Вопрос 1.2 (выбор ответа).** Что произойдёт при создании объекта `ExperimentConfig(name="test", learning_rate=0)`?

- A. Объект создастся, ошибка возникнет при обучении
- B. Объект создастся, значение `learning_rate` будет заменено на значение по умолчанию
- C. Возникнет исключение `ValueError` в методе `__post_init__`
- D. Возникнет исключение `TypeError`

**Вопрос 1.3 (выбор ответа).** Какой термин описывает свойство, при котором код, работающий с объектами базового класса, корректно работает и с объектами подклассов?

- A. Инкапсуляция
- B. Наследование
- C. Полиморфизм
- D. Абстракция

**Вопрос 1.4 (выбор ответа).** Что возвращает `EXPERIMENT_REGISTRY["distilbert"]`?

- A. Объект `DistilBertExperiment`
- B. Класс `DistilBertExperiment`
- C. Строку `"distilbert"`
- D. Конфигурацию эксперимента

## Часть 2. Интерпретация предсказаний: LIME

### 2.1. Назначение метода

LIME (Local Interpretable Model-agnostic Explanations) объясняет отдельное предсказание классификатора путём возмущения входного экземпляра и аппроксимации поведения модели в окрестности этого экземпляра простой интерпретируемой моделью.

**Принцип работы.**

| Шаг | Действие |
|---|---|
| 1 | Входной текст подвергается возмущению: случайное подмножество слов удаляется |
| 2 | Модель вычисляет вероятности классов для каждого возмущённого текста |
| 3 | На парах «признаки возмущённого текста — вероятность выбранного класса» обучается разреженная линейная регрессия |
| 4 | Коэффициенты регрессии образуют объяснение: положительный коэффициент указывает на повышение вероятности класса, отрицательный — на понижение |

**Свойство модельной агностичности.** Метод требует только функцию, отображающую текст в вектор вероятностей, и не использует информацию о внутреннем устройстве модели. Это свойство делает LIME применимым к любой архитектуре, доступной в Hugging Face Transformers, без модификации кода интерпретации.

**Пример «до»: попытка объяснить предсказание без инструмента.**

In [ ]:
# Для этого раздела нам нужна обученная модель.
# Обучим быстрый baseline, чтобы дальше использовать его в LIME и Captum.
baseline_cfg = ExperimentConfig(name="baseline", model_name="distilbert-base-uncased", num_epochs=1)
baseline_exp = DistilBertExperiment(baseline_cfg)
baseline_result = baseline_exp.run(train_ds, eval_ds)
print(baseline_result)

from transformers import AutoModelForSequenceClassification as _AM
model = _AM.from_pretrained(f"./results/{baseline_cfg.name}", num_labels=4).to(device)
model.eval()

Пример ниже показывает наивный подход: удаляем по одному слову и смотрим, как меняется вероятность.

In [ ]:
text_naive = "Apple launches new iPhone at earnings event"
words = text_naive.split()

def predict_proba_single(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
    return torch.softmax(logits, dim=-1).cpu().numpy()[0]

base_proba = predict_proba_single(text_naive)[2]   # вероятность Business
print(f"base Business proba: {base_proba:.4f}")

for i, word in enumerate(words):
    reduced = " ".join(words[:i] + words[i+1:])
    proba = predict_proba_single(reduced)[2]
    print(f"без '{word}': {proba:.4f} (delta={proba - base_proba:+.4f})")

Проблема: перебор по одному слову не учитывает взаимодействия между словами. Удаление `Apple` и `iPhone` по отдельности может дать малый эффект, а их совместное отсутствие — большой.

**Пример «после»: LIME.**

In [ ]:
# !pip install -q lime
from lime.lime_text import LimeTextExplainer

def predict_proba(texts):
    inputs = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    ).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
    return torch.softmax(logits, dim=-1).cpu().numpy()

explainer = LimeTextExplainer(class_names=class_names, random_state=42)
text_lime = "Apple launches new iPhone at earnings event"
exp = explainer.explain_instance(text_lime, predict_proba, num_features=8, labels=[2])
for word, weight in exp.as_list(label=2):
    print(f"{word:15s} {weight:+.4f}")

LIME генерирует тысячи возмущений, включая удаление нескольких слов одновременно, и обучает линейную модель на этих возмущениях. Это позволяет уловить взаимодействия между словами.

### 2.2. Словарь терминов

| Термин | Определение |
|---|---|
| **Возмущение (perturbation)** | Модификация входного экземпляра, при которой часть признаков удаляется или заменяется. В текстовых задачах — удаление слов. |
| **Окрестность экземпляра** | Множество возмущённых вариантов исходного входа, полученных случайным удалением признаков. |
| **Модель-аппроксиматор** | Простая интерпретируемая модель (в LIME — линейная регрессия), обучаемая на возмущениях для приближения поведения чёрного ящика. |
| **Коэффициент регрессии** | Вес, с которым признак (слово) входит в линейную модель-аппроксиматор. Служит мерой важности признака для выбранного класса. |
| **Модельная агностичность** | Свойство метода интерпретации, при котором он не требует доступа к внутренним параметрам модели. |
| **Локальное объяснение** | Объяснение, описывающее поведение модели в окрестности одного экземпляра, без обобщения на корпус. |

### 2.3. Подготовка функции предсказания

LIME ожидает функцию, принимающую список строк и возвращающую матрицу вероятностей размера `(n_текстов, n_классов)`.

**Пример «до»: функция, принимающая один текст.**

In [ ]:
def predict_proba_single_bad(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
    return torch.softmax(logits, dim=-1).cpu().numpy()[0]

# Демонстрация некорректного поведения:
out = predict_proba_single_bad(["first text", "second text"])
print(type(out), out.shape)

Проблема: LIME передаёт **список** строк (сотни возмущений за раз), а функция принимает одну строку. При вызове `predict_proba_single(["a", "b"])` токенизатор обработает список как одну последовательность, что даст некорректный результат.

**Пример «после»: функция, принимающая список строк.**

In [ ]:
def predict_proba(texts):
    inputs = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    ).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
    return torch.softmax(logits, dim=-1).cpu().numpy()

**Что делает `predict_proba`.**

| Строка | Действие |
|---|---|
| `tokenizer(texts, ...)` | Токенизирует **список** строк, возвращая батч |
| `padding=True` | Выравнивает длину последовательностей в батче дополнением |
| `truncation=True` | Обрезает последовательности, превышающие `max_length` |
| `return_tensors="pt"` | Возвращает тензоры PyTorch |
| `.to(device)` | Перемещает тензоры на GPU или CPU |
| `torch.no_grad()` | Отключает вычисление градиентов, снижая потребление памяти |
| `model(**inputs).logits` | Прямой проход; `**inputs` распаковывает `input_ids` и `attention_mask` |
| `torch.softmax(..., dim=-1)` | Преобразует логиты в вероятности по измерению классов |
| `.cpu().numpy()` | Перемещает результат на CPU и конвертирует в numpy-массив |

**Изолированный запуск: проверка формы выхода.**

In [ ]:
sample_texts = [
    "Apple launches new iPhone",
    "Real Madrid wins the match",
    "Stock market rallies on earnings",
]

probs = predict_proba(sample_texts)
print(probs.shape)
print(probs.sum(axis=1))
print(probs.argmax(axis=1))

**Допишите код самостоятельно.** Реализуйте `predict_proba` для батча, в котором часть текстов пустая (`""`). Проверьте, что функция не падает и возвращает корректную форму. Подсказка: токенизатор Hugging Face обрабатывает пустые строки, добавляя специальные токены `[CLS]` и `[SEP]`.

In [ ]:
texts_with_empty = ["", "Apple launches new iPhone", ""]
probs_empty = predict_proba(texts_with_empty)
print(probs_empty.shape)
print(probs_empty)

### 2.4. Получение объяснения

In [ ]:
explainer = LimeTextExplainer(class_names=class_names, random_state=42)

text = "Apple launches new iPhone at earnings event"
explanation = explainer.explain_instance(
    text,
    predict_proba,
    num_features=8,
    labels=[2]
)

for word, weight in explanation.as_list(label=2):
    direction = "повышает" if weight > 0 else "понижает"
    print(f"{word:15s} {direction} вероятность класса Business на {abs(weight):.4f}")

**Что делает `explain_instance`.**

| Шаг | Действие |
|---|---|
| 1 | Разбивает `text` на слова |
| 2 | Генерирует `num_samples` возмущений (по умолчанию 5000): для каждого слова случайно решает, оставить его или удалить |
| 3 | Вызывает `predict_proba` на батче возмущений |
| 4 | Взвешивает возмущения по близости к исходному тексту |
| 5 | Обучает разреженную линейную регрессию на взвешенных данных |
| 6 | Возвращает объект `Explanation` с коэффициентами регрессии |

**Пояснение параметров `explain_instance`.**

| Параметр | Назначение |
|---|---|
| `text` | Объясняемый экземпляр |
| `predict_proba` | Функция, отображающая список строк в матрицу вероятностей |
| `num_features` | Число наиболее влиятельных слов в объяснении |
| `labels` | Список индексов классов, для которых строится объяснение |
| `num_samples` | Число генерируемых возмущений (по умолчанию 5000) |

**Интерпретация вывода.** Если модель ошиблась, разумно запросить объяснение для предсказанного класса: это позволяет понять, какие слова способствовали ошибочному решению.

**Изолированный запуск: объяснение для двух классов одновременно.**

In [ ]:
exp_both = explainer.explain_instance(
    text,
    predict_proba,
    num_features=5,
    labels=[1, 2],
)

print("Business (label=2):")
for word, weight in exp_both.as_list(label=2):
    print(f"  {word:15s} {weight:+.4f}")

print("\nSports (label=1):")
for word, weight in exp_both.as_list(label=1):
    print(f"  {word:15s} {weight:+.4f}")

**Изолированный запуск: извлечение вероятностей из объяснения.**

In [ ]:
local_pred = exp.predict_proba
print(local_pred)
print(predict_proba([text])[0])

**Допишите код самостоятельно.** Для ошибочного примера из вашего эксперимента получите объяснение LIME для предсказанного класса и для истинного класса. Сравните списки слов. Сформулируйте вывод о том, какие слова модель не учла при ошибочном предсказании.

### 2.5. Ограничения метода

| Ограничение | Описание | Практическое следствие |
|---|---|---|
| Локальность | Объяснение описывает поведение в окрестности одного экземпляра | Результат не обобщается на другие экземпляры или корпус |
| Стохастичность | Результат зависит от случайной выборки возмущений | При изменении `random_state` набор важных слов может измениться |
| Отсутствие гарантии верности | Коэффициенты линейной модели приближают поведение чёрного ящика | Для задач с высокой ценой ошибки рекомендуется сопоставление с градиентными методами |

**Рекомендация по числу возмущений.** Для текстовых данных рекомендуется использовать не менее 5000 возмущений (параметр `num_samples`), чтобы аппроксимация была устойчивой.

**Изолированный запуск: демонстрация стохастичности.**

In [ ]:
results_per_seed = []
for seed in [0, 1, 2]:
    expl = LimeTextExplainer(class_names=class_names, random_state=seed)
    e = expl.explain_instance(text, predict_proba, num_features=5, labels=[2])
    top_words = [w for w, _ in e.as_list(label=2)]
    results_per_seed.append(top_words)

for seed, words in enumerate(results_per_seed):
    print(f"seed={seed}: {words}")

**Допишите код самостоятельно.** Напишите функцию `stable_lime(text, n_runs=5)`, которая запускает LIME `n_runs` раз с разными `random_state`, усредняет веса слов и возвращает топ-5 слов по среднему абсолютному весу. Подсказка: соберите все пары `(word, weight)` в список, сгруппируйте по слову и усредните.

In [ ]:
def stable_lime(text, n_runs=5, num_features=8, label=2):
    all_weights = {}
    for seed in range(n_runs):
        expl = LimeTextExplainer(class_names=class_names, random_state=seed)
        e = expl.explain_instance(text, predict_proba, num_features=num_features, labels=[label])
        for word, weight in e.as_list(label=label):
            all_weights.setdefault(word, []).append(abs(weight))
    mean_weights = {w: float(np.mean(v)) for w, v in all_weights.items()}
    return sorted(mean_weights.items(), key=lambda x: x[1], reverse=True)[:5]

print(stable_lime(text, n_runs=3))

### 2.6. Задания и вопросы

**Задание 2.1.** Для одного и того же текста получите объяснения LIME при трёх различных значениях `random_state`. Сравните множества слов, попавших в топ-5 по абсолютной величине веса. Сформулируйте вывод о том, можно ли на основании одного запуска LIME утверждать, что модель «смотрит на слово X».

**Задание 2.2.** Изучите параметр `num_samples` в документации `LimeTextExplainer`. Увеличьте его значение до 5000 и проверьте, становится ли результат более стабильным между запусками.

**Задание 2.3.** Для ошибочного примера из вашего эксперимента получите объяснение LIME для предсказанного класса и для истинного класса. Сравните списки слов. Сформулируйте вывод о том, какие слова модель не учла при ошибочном предсказании.

**Вопрос 2.1 (выбор ответа).** Что представляет собой коэффициент регрессии в модели-аппроксиматоре LIME?

- A. Вероятность класса при отсутствии данного слова
- B. Меру важности слова для выбранного класса
- C. Число вхождений слова в обучающий корпус
- D. Градиент функции потерь по эмбеддингу слова

**Вопрос 2.2 (выбор ответа).** Какое свойство LIME обеспечивает его применимость к моделям разных архитектур?

- A. Локальность объяснения
- B. Использование градиентов модели
- C. Модельная агностичность
- D. Разреженность линейной регрессии

**Вопрос 2.3 (выбор ответа).** Почему результат LIME может измениться при повторном запуске с тем же текстом?

- A. Модель переобучается между запусками
- B. Возмущения выбираются случайно
- C. Токенизатор изменяет разбиение на слова
- D. Веса линейной регрессии инициализируются случайно

**Вопрос 2.4 (выбор ответа).** Какую форму имеет выход функции `predict_proba`, ожидаемой LIME?

- A. `(n_текстов,)` — вектор предсказанных классов
- B. `(n_текстов, n_классов)` — матрица вероятностей
- C. `(n_классов,)` — распределение для одного текста
- D. `(n_текстов, max_length)` — матрица идентификаторов токенов

## Часть 3. Интерпретация внутренних представлений: Captum

### 3.1. Назначение метода

Captum предоставляет реализации методов атрибуции для моделей PyTorch. **Integrated Gradients** вычисляет вклад каждого входного признака в выход модели путём интегрирования градиентов вдоль прямолинейного пути от опорного входа к фактическому.

**Принцип работы.**

| Шаг | Действие |
|---|---|
| 1 | Задаётся опорный вход (baseline) — тензор, для которого выход модели известен и интерпретируем |
| 2 | Строится прямолинейный путь в пространстве входов от опорного входа к фактическому |
| 3 | Вычисляется градиент выхода по входу в каждой точке пути |
| 4 | Градиенты интегрируются вдоль пути; результат умножения на разность входов образует атрибуцию |

**Свойство полноты (completeness).** Сумма атрибуций по всем признакам равна разности между выходом модели на фактическом входе и выходом на опорном входе. Это свойство позволяет контролировать качество аппроксимации: возвращаемое значение `delta` представляет собой ошибку аппроксимации интеграла.

**Применение к текстовым моделям.** В качестве входа выступают идентификаторы токенов, а атрибуция вычисляется на уровне эмбеддингов: метод показывает, какая часть изменения выходного логита обусловлена изменением эмбеддинга каждого токена.

**Пример «до»: попытка объяснить через веса внимания.**

In [ ]:
text_attn = "Apple launches new iPhone at earnings event"
inputs_attn = tokenizer(text_attn, return_tensors="pt", truncation=True, max_length=128).to(device)

outputs = model(**inputs_attn, output_attentions=True)
attn = outputs.attentions[-1]
print(attn.shape)
mean_attn = attn.mean(dim=1).squeeze(0)
cls_attn = mean_attn[0]
print(cls_attn.shape)

Проблема: веса внимания показывают, куда «смотрит» модель, но не показывают, **как это влияет на выход**. Токен может получать большое внимание, но не влиять на логит выбранного класса. Кроме того, веса внимания в разных головах разных слоёв интерпретируются по-разному.

**Пример «после»: Integrated Gradients.**

In [ ]:
# !pip install -q captum
from captum.attr import LayerIntegratedGradients

def forward_func(input_ids, attention_mask):
    return model(input_ids, attention_mask=attention_mask).logits

lig = LayerIntegratedGradients(forward_func, model.distilbert.embeddings)

inputs = tokenizer(text_lime, return_tensors="pt", truncation=True, max_length=128).to(device)
attributions, delta = lig.attribute(
    inputs=inputs["input_ids"],
    target=2,
    additional_forward_args=(inputs["attention_mask"],),
    n_steps=50,
    return_convergence_delta=True,
)
print(attributions.shape)
print(delta)

Captum вычисляет вклад каждого токена в логит выбранного класса, а не в веса внимания. Это отвечает на вопрос «что повлияло на решение», а не «куда смотрела модель».

### 3.2. Словарь терминов

| Термин | Определение |
|---|---|
| **Атрибуция** | Присвоение каждому входному признаку числовой меры его вклада в выход модели. |
| **Опорный вход (baseline)** | Входной тензор, относительно которого вычисляется атрибуция. Для слоя эмбеддингов по умолчанию используется нулевой тензор. |
| **Интегрирование градиентов** | Вычисление интеграла градиентов вдоль пути от опорного входа к фактическому. |
| **Слой атрибуции** | Слой модели, на котором вычисляются градиенты и атрибуции. В пособии — слой эмбеддингов. |
| **Свойство полноты** | Требование, согласно которому сумма атрибуций равна разности выходов на фактическом и опорном входах. |
| **`delta` (ошибка аппроксимации)** | Разность между суммой атрибуций и теоретическим значением, предсказанным свойством полноты. Малое значение указывает на надёжность аппроксимации. |
| **`n_steps`** | Число шагов аппроксимации интеграла. Увеличение повышает точность и вычислительную стоимость. |

### 3.3. Вычисление атрибуции

Для применения `LayerIntegratedGradients` необходимо указать модель, слой, к которому применяется атрибуция, и функцию прямого прохода, возвращающую логиты.

**Расположение слоя эмбеддингов.** В модели `AutoModelForSequenceClassification` из Transformers слой эмбеддингов доступен как `model.distilbert.embeddings`. Для модели RoBERTa путь отличается: `model.roberta.embeddings`. Способ определения пути — изучение структуры модели через `print(model)`.

**Изолированный запуск: поиск слоя эмбеддингов.**

In [ ]:
model_for_path = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=4
)
print(model_for_path)

После обнаружения пути `distilbert.embeddings` обращение к слою: `model.distilbert.embeddings`.

In [ ]:
from captum.attr import LayerIntegratedGradients

def forward_func(input_ids, attention_mask):
    return model(input_ids, attention_mask=attention_mask).logits

lig = LayerIntegratedGradients(forward_func, model.distilbert.embeddings)

inputs = tokenizer(text_lime, return_tensors="pt", truncation=True, max_length=128).to(device)
attributions, delta = lig.attribute(
    inputs=inputs["input_ids"],
    target=2,
    additional_forward_args=(inputs["attention_mask"],),
    n_steps=50,
    return_convergence_delta=True,
)

**Что делает `forward_func`.** Принимает `input_ids` и `attention_mask`, выполняет прямой проход модели и возвращает тензор логитов формы `(batch, n_classes)`. Captum вызывает эту функцию многократно — по одному разу на каждый шаг интегрирования, поэтому она должна быть быстрой и не содержать побочных эффектов.

**Что делает `lig.attribute`.**

| Шаг | Действие |
|---|---|
| 1 | Создаёт `n_steps + 1` точек на прямолинейном пути от нулевого тензора к `inputs` |
| 2 | Для каждой точки вычисляет градиент логита `target` по выходу слоя эмбеддингов |
| 3 | Усредняет градиенты (аппроксимация интеграла методом трапеций) |
| 4 | Умножает усреднённый градиент на разность между `inputs` и baseline |
| 5 | Возвращает атрибуции формы `(batch, seq_len, embed_dim)` и `delta` |

**Пояснение параметров `attribute`.**

| Параметр | Назначение |
|---|---|
| `inputs` | Тензор идентификаторов токенов |
| `target` | Индекс класса, для которого вычисляется атрибуция |
| `additional_forward_args` | Дополнительные аргументы, передаваемые в `forward_func` (в данном случае — маска внимания) |
| `n_steps` | Число шагов аппроксимации интеграла |
| `return_convergence_delta` | Требование вернуть ошибку аппроксимации |

**Интерпретация `delta`.** Малое значение `delta` указывает на то, что аппроксимация надёжна. При большом значении следует увеличить `n_steps`.

**Изолированный запуск: проверка формы выхода.**

In [ ]:
print(inputs["input_ids"].shape)
print(attributions.shape)
print(delta)
print(f"delta = {delta.item():.6f}")

**Допишите код самостоятельно.** Напишите функцию `attribute_example(model, tokenizer, text, target_class, n_steps=50)`, которая:
1. токенизирует текст;
2. создаёт `LayerIntegratedGradients` для слоя эмбеддингов;
3. вызывает `attribute`;
4. возвращает кортеж `(attributions, delta, tokens)`.

Подсказка: путь к слою эмбеддингов зависит от модели. Передайте его вторым аргументом функции.

In [ ]:
def attribute_example(model, tokenizer, text, target_class, embedding_layer, n_steps=50, device=device):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128).to(device)
    def fwd(input_ids, attention_mask):
        return model(input_ids, attention_mask=attention_mask).logits
    lig_local = LayerIntegratedGradients(fwd, embedding_layer)
    attrs, delta_local = lig_local.attribute(
        inputs=inputs["input_ids"],
        target=target_class,
        additional_forward_args=(inputs["attention_mask"],),
        n_steps=n_steps,
        return_convergence_delta=True,
    )
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    return attrs, delta_local, tokens

attrs, d, toks = attribute_example(model, tokenizer, text_lime, 2, model.distilbert.embeddings)
print(attrs.shape, d.item(), len(toks))

### 3.4. Агрегация и интерпретация

Результат `attributions` имеет размерность `(batch, seq_len, embed_dim)`. Для получения вклада каждого токена выполняется суммирование по измерению эмбеддинга.

In [ ]:
attr = attributions.sum(dim=-1).squeeze(0).cpu().numpy()
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

pairs = sorted(zip(tokens, attr), key=lambda x: abs(x[1]), reverse=True)
for token, score in pairs[:8]:
    print(f"{token:15s} {score:+.4f}")

**Что делает `attributions.sum(dim=-1)`.** Суммирует атрибуции по измерению эмбеддинга. Результат: тензор формы `(batch, seq_len)`, где каждое значение — суммарный вклад соответствующего токена в логит выбранного класса.

**Что делает `squeeze(0)`.** Удаляет измерение батча (размер 1). Результат: вектор формы `(seq_len,)`.

**Что делает `sorted(zip(tokens, attr), key=lambda x: abs(x[1]), reverse=True)`.** Сортирует пары `(токен, атрибуция)` по абсолютной величине атрибуции в порядке убывания. Абсолютное значение используется потому, что важны как положительные, так и отрицательные вклады.

**Интерпретация знака.**

| Знак атрибуции | Значение |
|---|---|
| Положительный | Токен повышает логит выбранного класса |
| Отрицательный | Токен понижает логит выбранного класса |

**Сопоставление с LIME.** Сопоставление списков, полученных LIME и Captum, служит проверкой согласованности: если оба метода указывают на одни и те же токены, интерпретация приобретает дополнительный вес.

**Изолированный запуск: визуализация атрибуций по токенам.**

In [ ]:
import matplotlib.pyplot as plt

n_show = min(20, len(tokens))
fig, ax = plt.subplots(figsize=(12, 3))
colors = ["coral" if v > 0 else "steelblue" for v in attr[:n_show]]
ax.bar(range(n_show), attr[:n_show], color=colors)
ax.set_xticks(range(n_show))
ax.set_xticklabels(tokens[:n_show], rotation=45, ha="right")
ax.axhline(0, color="black", linewidth=0.5)
ax.set_ylabel("Атрибуция")
ax.set_title("Вклад токенов в логит класса (target=2)")
plt.tight_layout()
plt.show()

**Допишите код самостоятельно.** Напишите функцию `aggregate_subwords(tokens, attr)`, которая объединяет атрибуции субтокенов в рамках одного слова. Подсказка: субтокены RoBERTa начинаются с символа `Ġ` (пробел перед словом). Слова начинаются с `Ġ`; все последующие субтокены до следующего `Ġ` относятся к тому же слову. Функция должна вернуть список пар `(word, aggregated_attr)`.

In [ ]:
def aggregate_subwords(tokens, attr):
    words = []
    current_word = None
    current_sum = 0.0
    for tok, val in zip(tokens, attr):
        if tok.startswith("Ġ") or current_word is None:
            if current_word is not None:
                words.append((current_word, current_sum))
            current_word = tok.replace("Ġ", "")
            current_sum = float(val)
        else:
            current_word += tok
            current_sum += float(val)
    if current_word is not None:
        words.append((current_word, current_sum))
    return words

print(aggregate_subwords(tokens, attr)[:10])

### 3.5. Специфика применения к Transformer-моделям

**Ограничение уровня атрибуции.** Атрибуция на слое эмбеддингов не отражает вклад внутренних слоёв внимания и полносвязных слоёв. Для более глубокого анализа Captum предоставляет методы атрибуции на произвольных слоях, однако их применение к многомерным выходам требует дополнительной агрегации и выходит за рамки данного пособия.

**Достаточность для задачи объяснения ошибки.** Для задачи объяснения ошибки классификации атрибуции на уровне эмбеддингов, как правило, достаточно, поскольку она отвечает на вопрос о том, какие входные токены модель сочла значимыми.

**Выбор опорного входа.** Опорный вход по умолчанию — нулевой тензор. Для слоя эмбеддингов это означает, что все токены представлены нулевыми векторами. Такая опорная точка допустима, хотя в литературе обсуждаются альтернативы, например использование эмбеддинга токена `<pad>`.

**Изолированный запуск: сравнение baseline.**

In [ ]:
attr_zero, delta_zero = lig.attribute(
    inputs=inputs["input_ids"],
    target=2,
    additional_forward_args=(inputs["attention_mask"],),
    n_steps=50,
    return_convergence_delta=True,
)

pad_id = tokenizer.pad_token_id
baseline_ids = torch.full_like(inputs["input_ids"], pad_id)
attr_pad, delta_pad = lig.attribute(
    inputs=inputs["input_ids"],
    baselines=baseline_ids,
    target=2,
    additional_forward_args=(inputs["attention_mask"],),
    n_steps=50,
    return_convergence_delta=True,
)

print(f"delta (zero baseline): {delta_zero.item():.6f}")
print(f"delta (pad baseline):  {delta_pad.item():.6f}")

**Что делает `baselines=baseline_ids`.** Задаёт опорный вход явно. Captum строит путь от `baseline_ids` к `inputs["input_ids"]`. Если baseline не задан, используется нулевой тензор той же формы.

**Допишите код самостоятельно.** Сравните топ-5 токенов по атрибуции для двух baseline. Совпадают ли они? Сформулируйте вывод о чувствительности метода к выбору опорного входа.

### 3.6. Задания и вопросы

**Задание 3.1.** Для одного и того же ошибочного примера получите список токенов с наибольшими абсолютными значениями атрибуции Captum и список слов с наибольшими весами LIME. Вычислите размер пересечения двух множеств (например, долю слов из топ-5 LIME, присутствующих в топ-10 Captum). Сформулируйте вывод о согласованности методов для данного примера.

*Ориентир.* Обратите внимание, что LIME оперирует словами, а Captum — токенами субсловной разбивки. Для корректного сопоставления может потребоваться агрегация атрибуций субтокенов в рамках одного слова с помощью `tokenizer.convert_tokens_to_string`.

**Задание 3.2.** Исследуйте зависимость значения `delta` от параметра `n_steps`. Постройте таблицу значений `delta` для `n_steps` ∈ {10, 25, 50, 100}. Сформулируйте рекомендацию по выбору `n_steps`.

**Задание 3.3.** Определите путь к слою эмбеддингов для модели RoBERTa. Сравните атрибуции, полученные для DistilBERT и RoBERTa на одном и том же примере. Сформулируйте вывод о различии в значимых токенах.

**Вопрос 3.1 (выбор ответа).** Что представляет собой опорный вход (baseline) в методе Integrated Gradients?

- A. Текст, на котором модель обучена
- B. Входной тензор, относительно которого вычисляется атрибуция
- C. Средний вход по обучающему набору
- D. Вход, для которого модель даёт максимальную вероятность

**Вопрос 3.2 (выбор ответа).** Какое свойство Integrated Gradients позволяет контролировать качество аппроксимации?

- A. Локальность
- B. Модельная агностичность
- C. Свойство полноты
- D. Разреженность

**Вопрос 3.3 (выбор ответа).** Что означает отрицательное значение атрибуции для токена?

- A. Токен отсутствует в словаре
- B. Токен понижает логит выбранного класса
- C. Токен был обрезан при токенизации
- D. Токен относится к опорному входу

**Вопрос 3.4 (выбор ответа).** Какую форму имеет результат `lig.attribute` для текстового классификатора?

- A. `(batch, seq_len)` — вклад каждого токена
- B. `(batch, n_classes)` — логиты
- C. `(batch, seq_len, embed_dim)` — вклад каждого измерения эмбеддинга
- D. `(seq_len, embed_dim)` — без измерения батча

## Часть 4. Итоговый отчёт

### 4.1. Таблица сравнения экспериментов

Результаты, возвращённые объектами экспериментов, сводятся в таблицу с колонками `name`, `eval_accuracy`, `eval_loss`.

In [ ]:
import pandas as pd

results = [baseline_result]
df = pd.DataFrame(results)
print(df[["name", "eval_loss", "eval_accuracy"]].to_string(index=False))

**Назначение таблицы.** Таблица служит первым элементом отчёта и позволяет тимлиду оценить различия между конфигурациями.

**Изолированный запуск: форматирование таблицы с округлением.**

In [ ]:
df_fmt = df.copy()
df_fmt["eval_accuracy"] = df_fmt["eval_accuracy"].round(4)
df_fmt["eval_loss"] = df_fmt["eval_loss"].round(4)
try:
    print(df_fmt[["name", "eval_accuracy", "eval_loss"]].to_markdown(index=False))
except ImportError:
    print(df_fmt[["name", "eval_accuracy", "eval_loss"]].to_string(index=False))

**Что делает `to_markdown`.** Возвращает таблицу в формате Markdown, готовую для вставки в отчёт или README. Требует установленного пакета `tabulate`.

### 4.2. Визуализация объяснения

LIME предоставляет метод `show_in_notebook`, который отображает объяснение в виде цветной полосы с выделением слов, повышающих и понижающих вероятность выбранного класса.

**Способы сохранения.**

| Способ | Метод | Формат |
|---|---|---|
| HTML-файл | `explanation.save_to_file("lime.html")` | HTML |
| Изображение | Экспорт через `matplotlib` | PNG |
| Ноутбук | `explanation.show_in_notebook()` | Интерактивный виджет |

**Назначение визуализации.** Визуализация служит вторым элементом отчёта и даёт наглядное представление о том, на какие слова реагирует модель.

**Изолированный запуск: сохранение в HTML.**

In [ ]:
exp.save_to_file("lime_explanation.html")
print("Сохранено: lime_explanation.html")

**Изолированный запуск: горизонтальная диаграмма через matplotlib.**

In [ ]:
import matplotlib.pyplot as plt

pairs = exp.as_list(label=2)
words = [w for w, _ in pairs]
weights = [v for _, v in pairs]

fig, ax = plt.subplots(figsize=(8, 4))
colors = ["coral" if v > 0 else "steelblue" for v in weights]
ax.barh(range(len(words)), weights, color=colors)
ax.set_yticks(range(len(words)))
ax.set_yticklabels(words)
ax.axvline(0, color="black", linewidth=0.5)
ax.set_xlabel("Вес слова (LIME)")
ax.set_title("Вклад слов в предсказание класса Business")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig("lime_explanation.png", dpi=150)
plt.show()

**Что делает `ax.invert_yaxis()`.** Меняет порядок оси Y так, чтобы слово с наибольшим весом оказалось сверху. Без этого вызова `barh` рисует первый элемент внизу.

**Допишите код самостоятельно.** Напишите функцию `save_lime_figure(explanation, label, path)`, которая строит горизонтальную диаграмму, подписывает оси и сохраняет в PNG. Проверьте, что файл открывается и содержит все слова из `explanation.as_list(label=label)`.

In [ ]:
def save_lime_figure(explanation, label, path):
    pairs = explanation.as_list(label=label)
    words = [w for w, _ in pairs]
    weights = [v for _, v in pairs]
    fig, ax = plt.subplots(figsize=(8, max(3, 0.4 * len(words))))
    colors = ["coral" if v > 0 else "steelblue" for v in weights]
    ax.barh(range(len(words)), weights, color=colors)
    ax.set_yticks(range(len(words)))
    ax.set_yticklabels(words)
    ax.axvline(0, color="black", linewidth=0.5)
    ax.set_xlabel("Вес слова (LIME)")
    ax.set_title(f"Объяснение LIME для класса {label}")
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.close(fig)

save_lime_figure(exp, 2, "lime_figure.png")
print("saved lime_figure.png")

### 4.3. Текстовое заключение

Заключение формулируется как связный текст, отвечающий на вопрос о причине ошибки и о влиянии смены архитектуры.

**Пример структуры.**

> Baseline-модель DistilBERT достигает accuracy 0.89 на отложенной выборке. Основная категория ошибок — предсказание класса Sci/Tech для текстов класса Business. Анализ LIME для ошибочного примера показывает, что слова «iPhone» и «launches» повышают вероятность Sci/Tech, тогда как «earnings» понижает её, но недостаточно для исправления предсказания. Атрибуция Captum подтверждает, что токены, соответствующие «iPhone», имеют наибольший положительный вклад в логит Sci/Tech. Модель RoBERTa, обученная с теми же гиперпараметрами, достигает accuracy 0.91; для того же примера её объяснение LIME включает «earnings» среди слов, повышающих вероятность Business, что указывает на использование более широкого контекста.

**Требования к заключению.**

| Требование | Проверка |
|---|---|
| Указание метрик | Приведены значения accuracy для сравниваемых моделей |
| Указание категории ошибок | Названа пара классов, между которыми происходит путаница |
| Указание значимых слов | Приведены слова из объяснений LIME и Captum |
| Проверяемость | Каждое утверждение подкреплено результатами эксперимента |

**Изолированный запуск: автоматическая сборка черновика заключения.**

In [ ]:
from collections import Counter

def draft_report(df, errors, lime_top, captum_top):
    best = df.loc[df["eval_accuracy"].idxmax()]
    worst = df.loc[df["eval_accuracy"].idxmin()]
    pair_counts = Counter((t, p) for _, t, p in errors)
    (true_c, pred_c), count = pair_counts.most_common(1)[0]
    return (
        f"Лучшая модель: {best['name']} (accuracy {best['eval_accuracy']:.3f}). "
        f"Худшая: {worst['name']} (accuracy {worst['eval_accuracy']:.3f}). "
        f"Наиболее частая ошибка: класс {true_c} предсказан как {pred_c} "
        f"({count} случаев). LIME указывает на слова {lime_top}. "
        f"Captum подтверждает вклад токенов {captum_top}."
    )

print(draft_report(
    df_fmt,
    errors=[(0, 2, 3), (1, 2, 3), (2, 0, 2)],
    lime_top=["iPhone", "launches"],
    captum_top=["iPhone"],
))

**Что делает `draft_report`.** Находит строку с максимальной и минимальной точностью, определяет самую частую пару ошибок, подставляет переданные списки слов в шаблон. Результат — черновик, который студент редактирует вручную.

**Допишите код самостоятельно.** Расширьте `draft_report` так, чтобы она автоматически извлекала `lime_top` из объекта `Explanation` (первые три слова из `exp.as_list(label=pred_label)`) и `captum_top` из массива атрибуций (три токена с наибольшим абсолютным значением).

### 4.4. Задание

**Задание 4.1.** На основании полученных таблицы, визуализации и списков атрибуции сформулируйте три предложения, которые описывают: (1) различие в метриках между архитектурами, (2) характер ошибки baseline-модели на уровне слов, (3) изменение поведения при переходе к альтернативной архитектуре. Предложения должны быть проверяемыми на основании приведённых данных и не содержать утверждений, не подкреплённых результатами экспериментов.

**Задание 4.2.** Соберите все три элемента отчёта (таблица, PNG-диаграмма, текстовое заключение) в один Markdown-файл `report.md`. Проверьте, что таблица отображается, изображение вставляется через `![](lime_explanation.png)`, а заключение содержит ссылки на конкретные числа из таблицы.

In [ ]:
report_md = f"""# Отчёт по эксперименту

## Таблица сравнения

{df_fmt[["name", "eval_accuracy", "eval_loss"]].to_markdown(index=False)}

## Визуализация LIME

![LIME](lime_explanation.png)

## Заключение

(вставьте сюда три предложения из задания 4.1)
"""
with open("report.md", "w", encoding="utf-8") as f:
    f.write(report_md)
print("Сохранено: report.md")

## 5. Заключение

Пособие представило два подхода, дополняющих друг друга в практике экспериментирования с моделями обработки текста.

**Объектно-ориентированный интерфейс** на основе `dataclass` и `ABC` обеспечивает воспроизводимость и расширяемость серии экспериментов. Конфигурация отделена от исполнения, добавление новых архитектур не затрагивает существующий код.

**Инструменты интерпретации** LIME и Captum позволяют перейти от агрегированных метрик к анализу конкретных предсказаний, устанавливая соответствие между входными токенами и решением модели.

**Совместное применение** обоих подходов формирует основу для отчёта, который отвечает на вопросы не только о величине точности, но и о причинах ошибок и о механизме влияния архитектурных изменений.

## 7. Домашнее задание

Домашнее задание состоит из трёх частей и сдаётся в виде ноутбука с выполненными ячейками и кратким текстовым отчётом. Каждая часть опирается на материал соответствующего раздела пособия.

### 7.1. Часть A. Расширение интерфейса эксперимента

**Задание A1.** Реализуйте подкласс `BaseExperiment` для модели `bert-base-uncased`. Класс должен:

1. Наследовать `BaseExperiment`;
2. Реализовывать `build_model` и `run`;
3. Отличаться от `DistilBertExperiment` только значением `model_name` по умолчанию.

Проверьте, что цикл запуска из раздела 1.7 работает с тремя архитектурами (`distilbert`, `roberta`, `bert`) без изменений.

**Задание A2.** Реализуйте декоратор `@register_experiment("name")`, который автоматически добавляет класс в `EXPERIMENT_REGISTRY` при объявлении. Декоратор принимает строковый ключ и возвращает функцию, принимающую класс и возвращающую его же после записи в реестр. Примените декоратор к `DistilBertExperiment`, `RobertaExperiment` и `BertExperiment`. Убедитесь, что цикл запуска не изменился.

**Задание A3.** Добавьте в `ExperimentConfig` поле `warmup_ratio: float = 0.1` и проверку в `__post_init__`, что значение лежит в диапазоне `[0.0, 1.0]`. Передайте значение в `TrainingArguments` через параметр `warmup_ratio`. Запустите два эксперимента с `warmup_ratio=0.0` и `warmup_ratio=0.1`, сравните `eval_accuracy` и `eval_loss`.

**Задание A4.** Реализуйте подкласс `BaseExperiment`, который вычисляет макро-усреднённую F1-меру в дополнение к accuracy. Для этого:

1. Изучите параметр `compute_metrics` в документации `Trainer`;
2. Реализуйте функцию, принимающую объект `EvalPrediction` и возвращающую словарь с `accuracy` и `f1_macro`;
3. Передайте функцию в `Trainer` через параметр `compute_metrics`.

Сравните accuracy и F1-macro на отложенной выборке. Объясните расхождение, если оно есть.

### 7.2. Часть B. Интерпретация ошибок с помощью LIME

**Задание B1.** Обучите baseline-модель (`distilbert-base-uncased`, 1–3 эпохи) и модель `roberta-base` с теми же гиперпараметрами. Для каждой модели найдите по три ошибочных примера, в которых предсказанный класс отличается от истинного.

**Задание B2.** Для каждого ошибочного примера получите объяснение LIME для предсказанного класса и для истинного класса. Сохраните результаты в таблицу со столбцами:

| Модель | Текст (фрагмент) | Истинный класс | Предсказанный класс | Топ-3 LIME для предсказанного | Топ-3 LIME для истинного |
|---|---|---|---|---|---|

**Задание B3.** Сформулируйте вывод о том, какие слова модель не учла при ошибочном предсказании. Сравните поведение DistilBERT и RoBERTa: на каких примерах они ошибаются одинаково, а на каких — по-разному.

**Задание B4.** Проверьте устойчивость LIME: для одного примера запустите `explain_instance` пять раз с разными `random_state`. Усредните веса слов и сравните топ-5 по среднему абсолютному весу с результатом однократного запуска. Сформулируйте рекомендацию по числу запусков.

### 7.3. Часть C. Интерпретация внутренних представлений с помощью Captum

**Задание C1.** Для одного из ошибочных примеров из части B получите атрибуции `LayerIntegratedGradients` на слое эмбеддингов для предсказанного класса. Постройте горизонтальную диаграмму вклада первых 20 токенов.

**Задание C2.** Сравните топ-5 токенов по атрибуции Captum с топ-5 слов по весам LIME для того же примера. Вычислите размер пересечения двух множеств (например, долю слов из топ-5 LIME, присутствующих в топ-10 Captum). Сформулируйте вывод о согласованности методов.

**Задание C3.** Исследуйте зависимость значения `delta` от параметра `n_steps`. Постройте таблицу значений `delta` для `n_steps` ∈ {10, 25, 50, 100}. Постройте график зависимости `delta` от `n_steps`. Сформулируйте рекомендацию по выбору `n_steps` для отчёта.

**Задание C4.** Сравните атрибуции, полученные для DistilBERT и RoBERTa на одном и том же примере. Определите путь к слою эмбеддингов для RoBERTa (он отличается от `distilbert.embeddings`). Сформулируйте вывод о различии в значимых токенах между архитектурами.

### 7.4. Часть D. Итоговый отчёт

Соберите результаты частей A, B и C в один Markdown-файл `report.md` со следующими разделами:

1. **Таблица сравнения экспериментов.** Колонки: `name`, `eval_accuracy`, `eval_loss`, `f1_macro`. Строки — все запущенные конфигурации.
2. **Визуализация LIME.** PNG-диаграмма для одного ошибочного примера.
3. **Визуализация Captum.** PNG-диаграмма вклада токенов для того же примера.
4. **Текстовое заключение.** Три предложения, которые описывают:
   - различие в метриках между архитектурами;
   - характер ошибки baseline-модели на уровне слов;
   - изменение поведения при переходе к альтернативной архитектуре.

Каждое утверждение должно быть проверяемым на основании приведённых данных. Утверждения, не подкреплённые результатами экспериментов, не засчитываются.

### 7.5. Критерии оценивания

| Критерий | Вес |
|---|---|
| Часть A: интерфейс расширен, реестр работает без ветвлений, валидация конфига присутствует | 25% |
| Часть B: LIME применён к ошибочным примерам, таблица заполнена, выводы сформулированы | 25% |
| Часть C: Captum применён, `delta` исследован, сопоставление с LIME выполнено | 25% |
| Часть D: отчёт собран, содержит таблицу, две визуализации и проверяемое заключение | 25% |

**Штрафы:**

- Отсутствие запуска кода в ноутбуке — до 50% от веса части.
- Утверждения в заключении без ссылок на конкретные числа из таблицы — до 25% от веса части D.
- Использование `if model_name == ...` внутри цикла запуска — до 50% от веса части A.

## 8. Приложение. Сводная таблица терминов

| Термин | Раздел | Определение |
|---|---|---|
| Класс | 1.2 | Шаблон, описывающий набор атрибутов и методов |
| Объект | 1.2 | Конкретная сущность, созданная по шаблону класса |
| Атрибут | 1.2 | Именованное поле данных объекта или класса |
| Метод | 1.2 | Функция, определённая внутри класса |
| Конструктор | 1.2 | Метод `__init__`, вызываемый при создании объекта |
| Наследование | 1.2 | Получение подклассом атрибутов и методов базового класса |
| Переопределение метода | 1.2 | Реализация метода в подклассе с тем же именем |
| Полиморфизм | 1.2 | Единообразная обработка объектов разных подклассов |
| Абстрактный метод | 1.2 | Метод, объявленный без реализации, обязательный для подкласса |
| ABC | 1.2 | Абстрактный базовый класс, запрещающий инстанцирование без реализации методов |
| Инкапсуляция | 1.2 | Объединение данных и методов в одном классе |
| Контракт | 1.2 | Набор обязательств, задаваемых абстрактными методами |
| Конфигурация | 1.1 | Набор параметров, полностью определяющий один запуск |
| Возмущение | 2.2 | Модификация входа удалением части признаков |
| Окрестность экземпляра | 2.2 | Множество возмущённых вариантов исходного входа |
| Модель-аппроксиматор | 2.2 | Простая интерпретируемая модель, приближающая поведение чёрного ящика |
| Коэффициент регрессии | 2.2 | Вес признака в линейной модели-аппроксиматоре |
| Модельная агностичность | 2.2 | Независимость метода от внутреннего устройства модели |
| Локальное объяснение | 2.2 | Объяснение поведения в окрестности одного экземпляра |
| Атрибуция | 3.2 | Числовая мера вклада признака в выход модели |
| Опорный вход | 3.2 | Входной тензор, относительно которого вычисляется атрибуция |
| Интегрирование градиентов | 3.2 | Вычисление интеграла градиентов вдоль пути от опорного входа |
| Слой атрибуции | 3.2 | Слой модели, на котором вычисляются атрибуции |
| Свойство полноты | 3.2 | Равенство суммы атрибуций разности выходов |
| `delta` | 3.2 | Ошибка аппроксимации интеграла |
| `n_steps` | 3.2 | Число шагов аппроксимации интеграла |